In [1]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, JSONLoader, CSVLoader, WebBaseLoader, ImageCaptionLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from glob import glob

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
urls = ['https://n.news.naver.com/article/082/0001362787?cds=news_media_pc&type=editn',
        'https://n.news.naver.com/article/658/0000132175?type=main']
webloader = WebBaseLoader(urls)
document = webloader.load() # PDF 파일을 읽어서 LangChain의 Document 리스트(페이지별 텍스트+메타데이터)로 변환
print( document )

[Document(metadata={'source': 'https://n.news.naver.com/article/082/0001362787?cds=news_media_pc&type=editn', 'title': '부산 초등 3학년 전원, 방과후 이용권 연 50만 원까지 부담 ‘0’', 'language': 'ko'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n부산 초등 3학년 전원, 방과후 이용권 연 50만 원까지 부담 ‘0’\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n본문 바로가기\n\n\n\n\n\n\n이전 페이지\n\n\n\n\n\n\n\n\n\n\n부산일보\n\n\n\n\n\n구독\n\n메인 뉴스판에서 부산일보 주요뉴스를  볼 수 있습니다.\n보러가기\n\n\n부산일보 언론사 구독 해지되었습니다.\n\n\n\n\n\n\n\n\n\n\n주요뉴스\n이슈\n신문보기\n사설/칼럼\n정치\n사회\n경제\n생활\n세계\nIT\n랭킹\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n부산일보\n\n부산일보\n\n\n\nPICK\n안내\n\n\n언론사가 주요기사로선정한 기사입니다.\n언론사별 바로가기\n닫기\n\n\n\n\n부산 초등 3학년 전원, 방과후 이용권 연 50만 원까지 부담 ‘0’\n\n\n\n\n\n\n\n\n\n이상배 기자\n\n\n\n\n\n\n입력\n2026.01.15. 오전 10:10\n\n\n\n기사원문\n \n\n\n\n\n\n\n\n\n\n\n추천\n반응\n\n\n\n\n쏠쏠정보\n0\n\n\n\n\n흥미진진\n0\n\n\n\n\n공감백배\n0\n\n\n\n\n분석탁월\n0\n\n\n\n\n후속강추\n0\n\n\n \n\n\n\n\n댓글\n반응\n\n\n\n\n\n\n텍스트 음성 변환 서비스 사용하기\n\n\n\n성별\n남성\n여성\n\n\n말하기 속도\n느림\n보통\n빠름\n\n이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다.\n본문듣기

In [4]:
s = 'aa\n\n\n\nbb'
" ".join(s.split())

'aa bb'

In [ ]:
# web에서 가져오면 클리닝 작업 해주기
def clean_newlines(text):
    return ' '.join(text.split())

for doc in document:
    doc.page_content = clean_newlines(doc.page_content) # 공백 or 개행 문자 제거

In [8]:
document

[Document(metadata={'source': 'https://n.news.naver.com/article/082/0001362787?cds=news_media_pc&type=editn', 'title': '부산 초등 3학년 전원, 방과후 이용권 연 50만 원까지 부담 ‘0’', 'language': 'ko'}, page_content='부산 초등 3학년 전원, 방과후 이용권 연 50만 원까지 부담 ‘0’ 본문 바로가기 이전 페이지 부산일보 구독 메인 뉴스판에서 부산일보 주요뉴스를 볼 수 있습니다. 보러가기 부산일보 언론사 구독 해지되었습니다. 주요뉴스 이슈 신문보기 사설/칼럼 정치 사회 경제 생활 세계 IT 랭킹 부산일보 부산일보 PICK 안내 언론사가 주요기사로선정한 기사입니다. 언론사별 바로가기 닫기 부산 초등 3학년 전원, 방과후 이용권 연 50만 원까지 부담 ‘0’ 이상배 기자 입력 2026.01.15. 오전 10:10 기사원문 추천 반응 쏠쏠정보 0 흥미진진 0 공감백배 0 분석탁월 0 후속강추 0 댓글 반응 텍스트 음성 변환 서비스 사용하기 성별 남성 여성 말하기 속도 느림 보통 빠름 이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다. 본문듣기 시작 닫기 글자 크기 변경하기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 SNS 보내기 인쇄하기 부산 모든 초등학교서 돌봄교실 확보오후 8시까지 탄력 돌봄 서비스 제공우리동네자람터 10곳→16곳 확대 해당 기사 내용을 활용해서 구글 생성형 AI ‘제미나이(Gemini)’로 만든 이미지. 이상배 기자 sangbae@busan.com올해부터 부산 지역 초등학교 3학년이라면 누구나 연간 50만 원 이내의 방과후 프로그램 이용권을 지원받아 유상 강좌를 무료로 수강할 수 있다. 기존 저학년 중심의 무상 방과후 지원에서 한 걸음 더 나아가, 교육 수요가 본격적으로 늘어나는 학년까지 지원 대상을 확대한 것이다. 이에 따라 학생들의 방과 후 시간의

In [3]:
len(document)

2

In [ ]:
# 문서 분할
txt_split = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=20) 
docs = txt_split.split_documents(document)

In [11]:
len(docs)

23

In [ ]:
# embedding 및 벡터 DB 생성
embedding = OpenAIEmbeddings( model='text-embedding-ada-002')
vdb = FAISS.from_documents( docs, embedding )

In [13]:
vdb.save_local('faiss_db')

| 파일명           | 역할 설명                                                                                                                                        |
| ------------- | -------------------------------------------------------------------------------------------------------------------------------------------- |
| `index.faiss` | **FAISS 인덱스 파일**<br>문서 임베딩 벡터를 저장하는 **바이너리 파일**입니다.<br>빠른 유사도 검색을 위한 **FAISS 엔진의 핵심 데이터 구조**가 포함되어 있습니다.                                     |
| `index.pkl`   | **메타데이터 저장 파일**<br>각 문서(Document)의 `page_content`, `metadata` 등 **문자 정보**를 Python 객체(pickle 형식)로 저장합니다.<br>FAISS 인덱스와 함께 문서를 다시 로드할 때 필요합니다. |

In [ ]:
# 프롬프트 템플릿 정의
template = """질문에 대해 아래의 제공된 문맥(context)만을 사용하여 답변하세요. 
답을 모른다면 모른다고 말하고 답변을 지어내지 마세요. 
최대한 세 문장 내로 간결하게 답변하세요.

Question: {question} 
Context: {context} 
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI( model='gpt-4o-mini', temperature=0)
retriever = vdb.as_retriever() # vdb.as_retriever(k=4) 

def format_docs(docs):
    # print( "문서:", docs.page_content)
    return "\n".join([doc.page_content for doc in docs])

# 유사도 검색
qa = ( 
        {"context": retriever | format_docs, "question": RunnablePassthrough()} #유사도 검색에서 반환된 문서들을 하나의 문자열로 변환
        | prompt  
        | llm  
        | StrOutputParser() 
    )
query = "부산 지역 초등학교 3학년이라면 방과후 이용권 얼마 받니? " # question
result = qa.invoke( query )
print( result )

부산 지역 초등학교 3학년 학생은 연간 50만 원 이내의 방과후 이용권을 지원받습니다. 이 이용권을 통해 유상 강좌를 무료로 수강할 수 있습니다. 따라서 모든 초등 3학년 학생이 가계 부담 없이 다양한 방과후 프로그램을 선택할 수 있습니다.
